# 小红书帖子 · 第一阶段探索分析（结构化步骤）

**主路径（推荐）**：下面「一键主流程」三格——环境 → `run_notebook(CFG)` → 看 `output/experiments/<run_id>/summary.md`。

**关系分析结论请以实验目录为准**：`output/experiments/<run_id>/`（含 `config.json`、`metrics.csv`、各类 `relation_v1_*.csv`）。Phase1 清洗表与图表在 `output/phase1/`。

---

**推荐**：在本仓库根目录启动 Jupyter（或让下面代码格自动探测根目录）。

```bash
cd /Users/yilin/project/2604-robotic_failure_research
jupyter notebook notebooks/phase1_structured.ipynb
```

**流水线对应代码**（脚本复现）：

| 步骤 | 模块 | 说明 |
|------|------|------|
| A | `phase1/preprocess.py` | 读表、合并 `post_category`、一级/二级去重、统一长表 |
| B | `phase1/features.py` | 角色/拟人/边界/玩梗词典特征 |
| C | `phase1/analysis.py` | 互动图、聚类、导出辅助 CSV/图 |
| D | `phase1/reports.py` | `data_quality`、摘要报告、codebook 草案 |
| 一键 | `phase1/notebook_one_click.py` 或 `python run_phase1.py` | Notebook 一键 / 终端全流程 |

## 一键主流程（推荐）

依次运行下面 **3 个代码格**：

1. **环境与路径**（`PROJECT_ROOT`、`XLSX_PATH`）
2. **配置并运行**（只改 `CFG`，调用 `run_notebook(CFG)`）
3. **查看结果**（`summary.md` + 总体表）

人类入口说明另见：`output/experiments/README.md`。

In [13]:
# 1) 环境与路径
from pathlib import Path
import os

_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "phase1" / "pipeline.py").exists()),
    _cwd,
)
if PROJECT_ROOT != _cwd:
    os.chdir(PROJECT_ROOT)

XLSX_PATH = PROJECT_ROOT / "data/2604-小红书/小红书帖子数据.xlsx"
# XLSX_PATH = Path("/绝对路径/小红书帖子数据.xlsx")

assert (PROJECT_ROOT / "phase1" / "pipeline.py").exists(), "未找到本项目根目录（需含 phase1/pipeline.py）"
assert XLSX_PATH.is_file(), f"找不到 Excel：{XLSX_PATH.resolve()}"

os.environ["ROBOTIC_FAILURE_XLSX"] = str(XLSX_PATH.resolve())

print("ROOT =", PROJECT_ROOT)
print("XLSX =", XLSX_PATH.resolve())

ROOT = /Users/yilin/project/2604-robotic_failure_research
XLSX = /Users/yilin/project/2604-robotic_failure_research/data/2604-小红书/小红书帖子数据.xlsx


## 2) 配置并一键运行

只改 **`CFG`**。`run_notebook` 会依次：`run_phase1_pipeline`（清洗、特征、图、报告）→ `relation_v1` 实验（结果在 `output/experiments/<run_id>/`）。

如需把 `relation_v1_*.csv` 再复制一份到 `output/phase1/data/`，设 `mirror_to_phase1_data=True`。

In [14]:
from phase1.notebook_one_click import NotebookRunConfig, run_notebook

CFG = NotebookRunConfig(
    xlsx_path=XLSX_PATH,
    min_chars=5,
    drop_repeated_single_char=True,
    run_phase1_full=True,
    run_relation_v1=True,
    mirror_to_phase1_data=False,
    run_topics=True,
    verbose=True,
    # run_id="2026-05-09_my_run",  # 可选：固定 run 名
)

RESULT = run_notebook(CFG)
RESULT

ModuleNotFoundError: No module named 'jieba'

In [9]:
# 3) 查看结果（关系实验）
from IPython.display import Markdown, display
import pandas as pd

if "run_dir" in RESULT:
    display(Markdown((RESULT["run_dir"] / "summary.md").read_text(encoding="utf-8")))
    display(pd.read_csv(RESULT["overall_csv"]))
    print("run_dir:", RESULT["run_dir"])
    print("audit:", RESULT.get("audit_csv"))
else:
    print("未运行 relation_v1。统一表路径:", RESULT.get("unified_csv"))

过滤后一级评论: 16808
过滤后二级评论: 14437


---

## （可选）分步流程与汇报、LDA 等

以下为历史分步单元：若已用上面一键跑通，可跳过；需要单独导出 `output/data/` 汇报表或做 LDA 时再跑。

### 1.1 预处理阶段汇报表（仅步骤 A）

下面两格依赖 `main_df, merged, l1, l2, unified...`。若只跑了一键且未保留变量，请先在一键格后执行：

`from phase1.pipeline import run_phase1_pipeline`  
`result = run_phase1_pipeline(xlsx=XLSX_PATH, min_chars=CFG.min_chars, drop_repeated_single_char=CFG.drop_repeated_single_char)`  
再解包 `main_df = result["main_df"]` 等。


In [15]:
from pathlib import Path
import pandas as pd

REPORT_OUT = PROJECT_ROOT / "output" / "data"
REPORT_OUT.mkdir(parents=True, exist_ok=True)

unified_after_filter = unified

# 阶段快照（仅预处理，不含词表/特征）
stage_rows = [
    {
        "stage": "A0_raw_main",
        "description": "原始主表（小红书帖子数据）",
        "rows": len(main_df),
        "unique_posts": main_df["帖子id"].nunique() if "帖子id" in main_df.columns else pd.NA,
        "unique_comment_id": pd.NA,
        "comment_level": pd.NA,
        "missing_content": pd.NA,
        "missing_content_rate": pd.NA,
    },
    {
        "stage": "A1_merged_category",
        "description": "主表合并 post_category（含按帖子id覆写）",
        "rows": len(merged),
        "unique_posts": merged["帖子id"].nunique() if "帖子id" in merged.columns else pd.NA,
        "unique_comment_id": pd.NA,
        "comment_level": pd.NA,
        "missing_content": pd.NA,
        "missing_content_rate": pd.NA,
    },
    {
        "stage": "A2_l1_dedup",
        "description": "一级评论去重后（未过滤）",
        "rows": len(l1),
        "unique_posts": l1["帖子id"].nunique() if "帖子id" in l1.columns else pd.NA,
        "unique_comment_id": l1["comment_id"].nunique() if "comment_id" in l1.columns else pd.NA,
        "comment_level": 1,
        "missing_content": l1["content"].isna().sum() if "content" in l1.columns else pd.NA,
        "missing_content_rate": (l1["content"].isna().mean() if "content" in l1.columns else pd.NA),
    },
    {
        "stage": "A3_l2_dedup",
        "description": "二级评论去重后（未过滤）",
        "rows": len(l2),
        "unique_posts": l2["帖子id"].nunique() if "帖子id" in l2.columns else pd.NA,
        "unique_comment_id": l2["comment_id"].nunique() if "comment_id" in l2.columns else pd.NA,
        "comment_level": 2,
        "missing_content": l2["content"].isna().sum() if "content" in l2.columns else pd.NA,
        "missing_content_rate": (l2["content"].isna().mean() if "content" in l2.columns else pd.NA),
    },
    {
        "stage": "A4_unified_before_filter",
        "description": "一级+二级统一评论长表（过滤前）",
        "rows": len(unified_before_filter),
        "unique_posts": unified_before_filter["帖子id"].nunique() if "帖子id" in unified_before_filter.columns else pd.NA,
        "unique_comment_id": unified_before_filter["comment_id"].nunique() if "comment_id" in unified_before_filter.columns else pd.NA,
        "comment_level": "1+2",
        "missing_content": ((unified_before_filter["content"] == "").sum() if "content" in unified_before_filter.columns else pd.NA),
        "missing_content_rate": ((unified_before_filter["content"] == "").mean() if "content" in unified_before_filter.columns else pd.NA),
    },
    {
        "stage": "A5_unified_after_filter",
        "description": "统一评论长表（过滤后：空值/最小长度/重复字符）",
        "rows": len(unified_after_filter),
        "unique_posts": unified_after_filter["帖子id"].nunique() if "帖子id" in unified_after_filter.columns else pd.NA,
        "unique_comment_id": unified_after_filter["comment_id"].nunique() if "comment_id" in unified_after_filter.columns else pd.NA,
        "comment_level": "1+2",
        "missing_content": ((unified_after_filter["content"] == "").sum() if "content" in unified_after_filter.columns else pd.NA),
        "missing_content_rate": ((unified_after_filter["content"] == "").mean() if "content" in unified_after_filter.columns else pd.NA),
    },
    {
        "stage": "A6_l1_after_filter",
        "description": "一级评论（过滤后）",
        "rows": len(l1_filtered),
        "unique_posts": l1_filtered["帖子id"].nunique() if "帖子id" in l1_filtered.columns else pd.NA,
        "unique_comment_id": l1_filtered["comment_id"].nunique() if "comment_id" in l1_filtered.columns else pd.NA,
        "comment_level": 1,
        "missing_content": ((l1_filtered["content"] == "").sum() if "content" in l1_filtered.columns else pd.NA),
        "missing_content_rate": ((l1_filtered["content"] == "").mean() if "content" in l1_filtered.columns else pd.NA),
    },
    {
        "stage": "A7_l2_after_filter",
        "description": "二级评论（过滤后）",
        "rows": len(l2_filtered),
        "unique_posts": l2_filtered["帖子id"].nunique() if "帖子id" in l2_filtered.columns else pd.NA,
        "unique_comment_id": l2_filtered["comment_id"].nunique() if "comment_id" in l2_filtered.columns else pd.NA,
        "comment_level": 2,
        "missing_content": ((l2_filtered["content"] == "").sum() if "content" in l2_filtered.columns else pd.NA),
        "missing_content_rate": ((l2_filtered["content"] == "").mean() if "content" in l2_filtered.columns else pd.NA),
    },
]

stage_summary = pd.DataFrame(stage_rows)

# 额外补充：关键键完整性/潜在问题 + 过滤效果
qc = {
    "counts_post_id_duplicated": (
        counts["帖子id"].duplicated().sum() if "帖子id" in counts.columns else pd.NA
    ),
    "l1_comment_id_duplicated_after_dedup": (
        l1["comment_id"].duplicated().sum() if "comment_id" in l1.columns else pd.NA
    ),
    "l2_comment_id_duplicated_after_dedup": (
        l2["comment_id"].duplicated().sum() if "comment_id" in l2.columns else pd.NA
    ),
    "l2_orphan_parent_comment_id": (
        (~l2["parent_comment_id"].isin(set(l1["comment_id"]))).sum()
        if {"parent_comment_id"}.issubset(l2.columns) and {"comment_id"}.issubset(l1.columns)
        else pd.NA
    ),
    "filtered_removed_total": len(unified_before_filter) - len(unified_after_filter),
    "filtered_removed_l1": (unified_before_filter["comment_level"].eq(1).sum() - l1_filtered.shape[0]),
    "filtered_removed_l2": (unified_before_filter["comment_level"].eq(2).sum() - l2_filtered.shape[0]),
}
qc_df = pd.DataFrame([qc])

# 导出汇报文件
stage_summary.to_csv(REPORT_OUT / "phase1_preprocess_stage_summary.csv", index=False)
qc_df.to_csv(REPORT_OUT / "phase1_preprocess_qc_summary.csv", index=False)

md_lines = [
    "# Phase1 预处理阶段汇报（步骤 A）",
    "",
    "## 阶段汇总",
    "```",
    stage_summary.to_string(index=False),
    "```",
    "",
    "## 关键键与去重检查 + 过滤效果",
    "```",
    qc_df.to_string(index=False),
    "```",
    "",
]
(REPORT_OUT / "phase1_preprocess_stage_summary.md").write_text("\n".join(md_lines), encoding="utf-8")

stage_summary

,stage,description,rows,unique_posts,unique_comment_id,comment_level,missing_content,missing_content_rate
0,A0_raw_main,原始主表（小红书帖子数据）,36867,16,<NA>,<NA>,<NA>,<NA>
1,A1_merged_category,主表合并 post_category（含按帖子id覆写）,36867,16,<NA>,<NA>,<NA>,<NA>
2,A2_l1_dedup,一级评论去重后（未过滤）,18893,16,18893,1,53,0.002805
3,A3_l2_dedup,二级评论去重后（未过滤）,17499,16,17499,2,161,0.009201
4,A4_unified_before_filter,一级+二级统一评论长表（过滤前）,36392,16,36392,1+2,214,0.00588
5,A5_unified_after_filter,统一评论长表（过滤后：空值/最小长度/重复字符）,31245,16,31245,1+2,0,0.0
6,A6_l1_after_filter,一级评论（过滤后）,16808,16,16808,1,0,0.0
7,A7_l2_after_filter,二级评论（过滤后）,14437,16,14437,2,0,0.0


## 2. 落盘清洗表（可选）

与 `python run_phase1.py` 输出路径一致：`output/phase1/`。

说明：
- `clean_l1_comments.csv` / `clean_l2_comments.csv`：去重后、未过滤
- `clean_comments_unified_before_filter.csv`：统一表过滤前
- `clean_comments_unified.csv`：统一表过滤后（空值、最小长度、重复字符规则已应用）
- `clean_l1_comments_filtered.csv` / `clean_l2_comments_filtered.csv`：过滤后按层级拆分

In [19]:
from phase1.config import OUT

# 创建子目录
EXPORT_DIR = OUT / "data"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 去重后（未过滤）
l1.to_csv(EXPORT_DIR / "clean_l1_comments.csv", index=False)
l2.to_csv(EXPORT_DIR / "clean_l2_comments.csv", index=False)

# 统一表过滤前后
unified_before_filter.to_csv(EXPORT_DIR / "clean_comments_unified_before_filter.csv", index=False)
unified.to_csv(EXPORT_DIR / "clean_comments_unified.csv", index=False)

# 过滤后分层
l1_filtered.to_csv(EXPORT_DIR / "clean_l1_comments_filtered.csv", index=False)
l2_filtered.to_csv(EXPORT_DIR / "clean_l2_comments_filtered.csv", index=False)

print("已写入", EXPORT_DIR)

已写入 /Users/yilin/project/2604-robotic_failure_research/output/phase1/data


## 3. 词典与特征（步骤 B）

词表定义见 `phase1/lexicons.py`；可在本格之后自行改词典再重跑。

In [ ]:
# 单一入口模式下，enriched 已由 run_phase1_pipeline 返回
enriched.head(2)

## 4. 分析导出（步骤 C）

含互动汇总、角色热力图、拟人共现、玩梗与边界、TF-IDF+KMeans 主题（轻量）。**首次画图**会配置中文字体（`phase1/config.py`）。

In [ ]:
from phase1.config import configure_matplotlib
from phase1.analysis import (
    ensure_dirs,
    boundary_outputs,
    interaction_map,
    meme_outputs,
    personhood_outputs,
    role_aggregate,
    topic_clusters,
)

configure_matplotlib()
ensure_dirs()

interaction_map(l1)
role_aggregate(enriched)
personhood_outputs(enriched)
meme_outputs(enriched)
boundary_outputs(enriched)
topic_clusters(
    unified["content"].tolist(),
    unified["post_category"].tolist(),
    top_n=min(8000, len(unified)),
)

enriched.to_csv(OUT / "comments_enriched.csv", index=False)
print("figures & csv -> output/phase1")

## 5. 报告（步骤 D）

In [ ]:
from phase1.reports import (
    build_phase1_summary,
    write_codebook_suggestions,
    write_data_quality,
)

write_data_quality(merged, l1, l2, unified)
build_phase1_summary(enriched, l1)
write_codebook_suggestions(enriched, l1)

## 6. 一键等价调用（可选）

与终端 `python run_phase1.py` 等价。

**重要说明**：`jieba` 不可用时，`char_bigram` 只能作为兜底实验，主题词会更碎、解释性更差。要得到稳定可解释结果，建议先安装并使用 `jieba` 或 `pkuseg`。

In [ ]:
# from phase1.pipeline import run_phase1_pipeline
# result = run_phase1_pipeline(xlsx=XLSX_PATH)  # 与上面定义的 XLSX_PATH 一致
# result["enriched"].head()

## 7. 可配置 LDA 主题建模（实验）

这一节在不替换现有 `TF-IDF+KMeans` 的前提下，新增一条更可解释的中文 LDA 流程：

- 使用 `CountVectorizer + sklearn LDA`
- 支持 tokenizer 切换（默认 `jieba`，可选 `pkuseg`）
- 停用词与噪音词来自可编辑配置文件：
  - `config/topic_modeling/general_stopwords.txt`
  - `config/topic_modeling/platform_noise_tokens.csv`
  - `config/topic_modeling/corpus_noise_tokens.csv`
  - `config/topic_modeling/domain_user_dict.txt`

你可以随时修改这些表后重跑本节。

In [ ]:
from phase1.topic_lda import (
    build_topic_documents,
    export_lda_result,
    load_topic_stopwords,
    run_lda,
    scan_lda_k,
)

# 你可以切换到 "pkuseg"（需先 pip install pkuseg）
TOKENIZER = "jieba"

stopwords = load_topic_stopwords()
print("stopwords/noise tokens:", len(stopwords))

docs = build_topic_documents(
    unified,
    tokenizer=TOKENIZER,
    min_tokens=3,
    stopwords=stopwords,
)
print("LDA usable docs:", len(docs), "/", len(unified))
docs[["content", "content_clean", "clean_token_count"]].head(5)

In [ ]:
# 先跑一个中等 K（建议从 10 开始）
lda_result = run_lda(
    unified,
    tokenizer=TOKENIZER,
    n_topics=10,
    min_df=10,
    max_df=0.5,
    max_features=8000,
    max_iter=30,
    sample_per_topic=30,
    top_terms=15,
)

lda_result.topics_terms.sort_values("topic")

In [ ]:
# 查看每个主题在 post_category 的分布
lda_result.topic_by_category.head(30)

In [ ]:
# 主题样本文本（最能代表主题的评论），用于人工可解释性判断
lda_result.topic_samples.head(50)

In [ ]:
# K 扫描：结合 perplexity + 主题大小均衡度做初筛，再人工看主题词表
k_diag = scan_lda_k(
    unified,
    tokenizer=TOKENIZER,
    k_values=(5, 8, 10, 12, 15),
    min_df=10,
    max_df=0.5,
)
k_diag

In [ ]:
# 导出 LDA 结果（与现有输出风格对齐）
export_lda_result(lda_result, prefix="lda", output_dir=OUT)
k_diag.to_csv(OUT / "lda_k_scan_diagnostics.csv", index=False)

print("已导出:")
print(OUT / "lda_topics_terms.csv")
print(OUT / "lda_topic_by_category.csv")
print(OUT / "lda_topic_samples.csv")
print(OUT / "lda_diagnostics.csv")
print(OUT / "lda_k_scan_diagnostics.csv")